# Download các thư viện cần thiết

In [2]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [3]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [4]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [5]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [6]:
def split_and_save_parquet(df, num_files, output_dir, type):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.{type}_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Chuẩn bị dữ liệu

In [7]:
df_user = read_parquet_user("./preprocessed-dataset")
df_user.head()

customer_id,gender,province,membership,created_date,last_sync_date,install_app,install_datetime,user_age_days,days_since_install,days_since_last_sync
i32,str,str,str,date,date,str,date,f64,f64,f64
9110357,"""female""","""Hồ Chí Minh""","""Standard""",2025-08-23,null,"""In-Store""",2025-08-23,38.596563,38.917091,null
9112155,"""female""","""Hà Nội""","""Standard""",2025-08-23,null,"""SPE""",2025-08-19,38.284309,42.917091,null
9111698,"""female""","""Hà Nội""","""Standard""",2025-08-23,null,"""SPE""",2025-08-23,38.375748,38.917091,null
9110836,"""male""","""Bình Dương""","""Standard""",2025-08-23,null,"""In-Store""",2025-08-23,38.509642,38.917091,null
9110747,"""female""","""Cà Mau""","""Standard""",2025-08-23,null,"""In-Store""",2025-08-23,38.524602,38.917091,null


In [8]:
df_transaction = read_parquet_transaction("./preprocessed-dataset")
df_transaction.head()

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]","decimal[38,4]",str,"decimal[38,4]"
"""0029130000021""",54400.0000,1,6672770,2024-05-26,"""In-Store""","""VietQR""",483,9600.0000,69000.0000,"""Bột ăn dặm""",0.2116
"""5506000000005""",112370.4735,1,3501018,2024-05-17,"""In-Store""","""VietQR""",854,6629.5265,119000.0000,"""Snack ăn dặm""",0.0557
"""0020250010001""",165000.0000,1,4899881,2024-05-26,"""In-Store""","""Tiền mặt""",376,20000.0000,185000.0000,"""Giặt xả cho bé""",0.1081
"""3047000000002""",43000.0000,1,6408731,2024-05-26,"""In-Store""","""VietQR""",161,0.0000,47000.0000,"""Dầu ăn & Gia vị""",0.0851
"""5140000000011""",270000.0000,1,5676745,2024-05-17,"""In-Store""","""VietQR""",412,0.0000,270000.0000,"""Bình sữa, phụ kiện""",0.0000


In [9]:
df_item = read_parquet_item("./preprocessed-dataset")
df_item.head()

item_id,price,category_l1,category_l2,brand,item_type,color,size,gender_target_final,description_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Con Cưng""","""Bộ quần áo""","""Không xác định""","""Không xác định""","""Bé Gái""","""Không xác định""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""﻿﻿Tã dán Merries size S 82 miế…","""[""Từ 4M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""﻿﻿﻿Bỉm tã quần Merries size M …","""12-36M"""


# Training Stage 1

In [10]:
import numpy as np
import pandas as pd
import polars as pl

from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GridSearchCV

from tqdm.auto import tqdm  # <- thêm dòng này


/datastore/uittogether/tools/miniconda3/envs/MABe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Chuẩn bị transaction, sort thời gian, tách train/valid (2024)

In [11]:
# df_transaction: Polars DataFrame gốc

df_trx = (
    df_transaction
    .select(["customer_id", "item_id", "created_date"])
    .drop_nulls(["customer_id", "item_id", "created_date"])
    .with_columns(
        pl.col("created_date")
        .cast(pl.Datetime)              # chuyển Date/String → Datetime
        .alias("created_datetime")
    )
    .sort("created_datetime")          # sắp xếp thời gian tăng dần
)

print("Tổng số dòng transaction:", df_trx.height)
print(df_trx.head())
print(df_trx.dtypes)

Tổng số dòng transaction: 35729825
shape: (5, 4)
┌─────────────┬───────────────┬──────────────┬─────────────────────┐
│ customer_id ┆ item_id       ┆ created_date ┆ created_datetime    │
│ ---         ┆ ---           ┆ ---          ┆ ---                 │
│ i32         ┆ str           ┆ date         ┆ datetime[μs]        │
╞═════════════╪═══════════════╪══════════════╪═════════════════════╡
│ 1028293     ┆ 4048000000008 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1158000000007 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1606000000010 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1627000000005 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 409015      ┆ 2803000000012 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
└─────────────┴───────────────┴──────────────┴─────────────────────┘
[Int32, String, Date, Datetime(time_unit='us', time_zone=None)]


In [12]:
# Chỉ lấy năm 2024
df_trx_2024 = df_trx.filter(
    pl.col("created_datetime").dt.year() == 2024
)

# Train = tháng 1 → 11/2024
df_train_pl = df_trx_2024.filter(
    pl.col("created_datetime").dt.month().is_between(1, 11, closed="both")
)

# Valid = tháng 12/2024
df_valid_pl = df_trx_2024.filter(
    pl.col("created_datetime").dt.month() == 12
)

print("Số dòng train (Polars):", df_train_pl.height)
print("Số dòng valid (Polars):", df_valid_pl.height)


Số dòng train (Polars): 32680632
Số dòng valid (Polars): 3049193


In [13]:
df_train = df_train_pl.to_pandas()
df_valid = df_valid_pl.to_pandas()

print("Số dòng train (pandas):", len(df_train))
print("Số dòng valid (pandas):", len(df_valid))
print(df_train.head())
print(df_valid.head())


Số dòng train (pandas): 32680632
Số dòng valid (pandas): 3049193
   customer_id        item_id created_date created_datetime
0      1028293  4048000000008   2024-01-01       2024-01-01
1       512190  1158000000007   2024-01-01       2024-01-01
2       512190  1606000000010   2024-01-01       2024-01-01
3       512190  1627000000005   2024-01-01       2024-01-01
4       409015  2803000000012   2024-01-01       2024-01-01
   customer_id        item_id created_date created_datetime
0      6981489  6382000000005   2024-12-01       2024-12-01
1      7092360  5427000000006   2024-12-01       2024-12-01
2      6274269  0007090000357   2024-12-01       2024-12-01
3      3192555  3436000000013   2024-12-01       2024-12-01
4      4177385  3953000000092   2024-12-01       2024-12-01


## Định nghĩa Estimator Stage 1 (ALS)

### Imports + config thư mục lưu

In [14]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, save_npz, load_npz
import joblib
from tqdm.auto import tqdm

from sklearn.base import BaseEstimator

# (khuyến nghị) giảm xung đột multi-thread giữa BLAS và implicit/OpenMP
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

ARTIFACT_DIR = Path("artifacts_stage1_als_gpu")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


### Helpers lưu/đọc JSON

In [15]:
def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


### Class Stage1 ALS (GPU) + persistence theo từng bước

In [16]:
from threadpoolctl import threadpool_limits

class ALSStage1CF(BaseEstimator):
    """
    Stage 1 Candidate Retrieval bằng Collaborative Filtering (ALS implicit) chạy GPU.

    Ý tưởng:
      1) Build user-item matrix có trọng số (log_count/rel_freq + ưu tiên category_l2 & age_group_final)
      2) (optional) BM25 weighting trước khi fit ALS
      3) Fit ALS trên GPU
      4) Lưu artifact theo từng bước:
         - ui_matrix.npz
         - mappings (user/item)
         - user_history, popular_items
         - model (CPU copy) + factors + config
         - metrics evaluate

    df_train: pandas DataFrame (train interactions)
    df_valid: pandas DataFrame (valid interactions)
    df_item : pandas DataFrame (item metadata: category_l2, age_group_final)

    Columns tối thiểu:
      - df_train: [user_col, item_col] (+ quantity optional)
      - df_valid: [user_col, item_col]
      - df_item : [item_col, category_l2, age_group_final] (missing -> fill UNK)
    """

    def __init__(
        self,
        df_train: pd.DataFrame,
        df_valid: pd.DataFrame,
        df_item: pd.DataFrame,
        user_col="customer_id",
        item_col="item_id",
        weight_type="log_count",          # "binary" | "count" | "log_count" | "rel_freq"
        alpha_cat=0.5,
        alpha_age=0.5,
        # ALS
        factors=128,
        iterations=30,
        regularization=0.05,
        alpha=2.0,
        # BM25 (trước ALS)
        use_bm25=True,
        bm25_k1=100,
        bm25_b=0.8,
        # Recommend/Eval
        k_eval=1000,
        use_tqdm=True,
        # Persistence
        save_dir="artifacts_stage1_als_gpu",
        run_name=None,
        # sklearn score config
        eval_filter=True,
        require_gpu=True,          # mặc định ép dùng GPU
        allow_cpu_fallback=False,  # nếu True thì tự chạy CPU khi GPU không có
    ):
        self.df_train = df_train
        self.df_valid = df_valid
        self.df_item = df_item

        self.user_col = user_col
        self.item_col = item_col

        self.weight_type = weight_type
        self.alpha_cat = alpha_cat
        self.alpha_age = alpha_age

        self.factors = factors
        self.iterations = iterations
        self.regularization = regularization
        self.alpha = alpha

        self.use_bm25 = use_bm25
        self.bm25_k1 = bm25_k1
        self.bm25_b = bm25_b

        self.k_eval = k_eval
        self.use_tqdm = use_tqdm

        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)

        self.run_name = run_name or time.strftime("%Y%m%d_%H%M%S")
        self.run_dir = self.save_dir / self.run_name
        self.run_dir.mkdir(parents=True, exist_ok=True)

        self.eval_filter = eval_filter

    # -------------------------
    # Persistence helpers
    # -------------------------
    def _save_config(self):
        cfg = {
            "user_col": self.user_col,
            "item_col": self.item_col,
            "weight_type": self.weight_type,
            "alpha_cat": self.alpha_cat,
            "alpha_age": self.alpha_age,
            "als": {
                "factors": self.factors,
                "iterations": self.iterations,
                "regularization": self.regularization,
                "alpha": self.alpha,
            },
            "bm25": {
                "use_bm25": self.use_bm25,
                "k1": self.bm25_k1,
                "b": self.bm25_b,
            },
            "k_eval": self.k_eval,
            "eval_filter": self.eval_filter,
            "run_name": self.run_name,
        }
        save_json(cfg, self.run_dir / "config.json")

    def _save_mappings(self):
        joblib.dump(self.user_index_to_id_, self.run_dir / "user_index_to_id.pkl")
        joblib.dump(self.item_index_to_id_, self.run_dir / "item_index_to_id.pkl")
        joblib.dump(self.user_id_to_index_, self.run_dir / "user_id_to_index.pkl")
        joblib.dump(self.item_id_to_index_, self.run_dir / "item_id_to_index.pkl")

    def _save_history_and_popularity(self):
        joblib.dump(self.user_history_, self.run_dir / "user_history.pkl")
        np.save(self.run_dir / "item_popularity.npy", self.item_popularity_)
        np.save(self.run_dir / "popular_item_indices.npy", self.popular_item_indices_)

    # -------------------------
    # Build weighted user-item CSR
    # -------------------------
    def _build_user_item_matrix(self):
        df = self.df_train.copy()

        cols = [self.user_col, self.item_col]
        if "quantity" in df.columns:
            cols.append("quantity")
        df = df[cols]

        if "quantity" in df.columns:
            agg = (
                df.groupby([self.user_col, self.item_col], as_index=False)
                  .agg(n_interactions=(self.item_col, "size"),
                       sum_qty=("quantity", "sum"))
            )
        else:
            agg = (
                df.groupby([self.user_col, self.item_col], as_index=False)
                  .agg(n_interactions=(self.item_col, "size"))
            )
            agg["sum_qty"] = agg["n_interactions"]

        # join item meta
        item_cols = [self.item_col, "category_l2", "age_group_final"]
        df_item_small = self.df_item[item_cols].drop_duplicates(subset=[self.item_col])
        agg = agg.merge(df_item_small, on=self.item_col, how="left")

        agg["category_l2"] = agg["category_l2"].fillna("__UNK_CAT2__")
        agg["age_group_final"] = agg["age_group_final"].fillna("__UNK_AGE__")

        base_raw = agg["sum_qty"].astype(float)

        if self.weight_type == "binary":
            base = np.ones(len(agg), dtype=np.float32)
        elif self.weight_type == "count":
            base = base_raw.to_numpy(dtype=np.float32)
        elif self.weight_type == "log_count":
            base = np.log1p(base_raw.to_numpy(dtype=np.float32))
        elif self.weight_type == "rel_freq":
            user_total = agg.groupby(self.user_col)["sum_qty"].transform("sum")
            base = (base_raw / user_total).to_numpy(dtype=np.float32)
        else:
            raise ValueError(f"Unknown weight_type: {self.weight_type}")

        agg["base_weight"] = base

        # preferences
        agg["user_total_qty"] = agg.groupby(self.user_col)["sum_qty"].transform("sum")
        agg["user_cat_l2_qty"] = agg.groupby([self.user_col, "category_l2"])["sum_qty"].transform("sum")
        agg["user_ageg_qty"] = agg.groupby([self.user_col, "age_group_final"])["sum_qty"].transform("sum")

        agg["cat_pref"] = (agg["user_cat_l2_qty"] / agg["user_total_qty"]).fillna(0.0)
        agg["age_pref"] = (agg["user_ageg_qty"] / agg["user_total_qty"]).fillna(0.0)

        factor = 1.0 + self.alpha_cat * agg["cat_pref"] + self.alpha_age * agg["age_pref"]
        w_ui = agg["base_weight"].to_numpy(dtype=np.float32) * factor.to_numpy(dtype=np.float32)
        agg["value"] = w_ui.astype(np.float32)

        # encode ids -> indices
        user_cat = agg[self.user_col].astype("category")
        item_cat = agg[self.item_col].astype("category")

        self.user_index_to_id_ = list(user_cat.cat.categories)
        self.item_index_to_id_ = list(item_cat.cat.categories)

        self.user_id_to_index_ = {uid: idx for idx, uid in enumerate(self.user_index_to_id_)}
        self.item_id_to_index_ = {iid: idx for idx, iid in enumerate(self.item_index_to_id_)}

        user_codes = user_cat.cat.codes.to_numpy()
        item_codes = item_cat.cat.codes.to_numpy()
        data = agg["value"].to_numpy(dtype=np.float32)

        n_users = len(self.user_index_to_id_)
        n_items = len(self.item_index_to_id_)

        ui = csr_matrix((data, (user_codes, item_codes)), shape=(n_users, n_items), dtype=np.float32)

        # user_history (train)
        # groupby user_code -> set item_code
        tmp = pd.DataFrame({"u": user_codes, "i": item_codes})
        self.user_history_ = tmp.groupby("u")["i"].apply(lambda s: set(s.to_numpy())).to_dict()

        # item popularity
        self.item_popularity_ = np.asarray(ui.sum(axis=0)).ravel()
        self.popular_item_indices_ = np.argsort(-self.item_popularity_)

        # persist
        save_npz(self.run_dir / "ui_matrix_raw.npz", ui)
        self._save_mappings()
        self._save_history_and_popularity()

        return ui

    def _fit_als(self, ui_matrix: csr_matrix):
        """
        Fit ALS:
        - Try GPU if available
        - Fallback CPU if GPU not available
        Persist:
        - ui_matrix_bm25.npz (if use_bm25)
        - als_model_cpu.pkl + factors
        - als_backend.json (gpu/cpu)
        """
        backend = {"backend": None, "reason": None}

        # 1) BM25 weighting (khuyến nghị theo implicit tutorial)
        if self.use_bm25:
            from implicit.nearest_neighbours import bm25_weight
            item_user = bm25_weight(ui_matrix.T.tocsr(), K1=self.bm25_k1, B=self.bm25_b)
            mat = item_user.T.tocsr()
            save_npz(self.run_dir / "ui_matrix_bm25.npz", mat)
        else:
            mat = ui_matrix

        self.user_item_matrix_ = mat  # dùng cho recommend + filter_already_liked_items

        # 2) Try GPU
        try:
            from implicit.gpu.als import AlternatingLeastSquares as GPUALS
            # nếu import được nhưng không có CUDA extension -> constructor sẽ raise ValueError
            self.als_gpu_ = GPUALS(
                factors=self.factors,
                iterations=self.iterations,
                regularization=self.regularization,
                alpha=self.alpha,
                calculate_training_loss=False,
            )
            self.als_gpu_.fit(mat)

            # convert to CPU for persistence if possible
            if hasattr(self.als_gpu_, "to_cpu"):
                self.als_cpu_ = self.als_gpu_.to_cpu()
            else:
                # fallback: không có to_cpu, sẽ dùng GPU model trực tiếp (có thể khó pickle)
                self.als_cpu_ = None

            backend["backend"] = "gpu"
            save_json(backend, self.run_dir / "als_backend.json")

            if self.als_cpu_ is not None:
                joblib.dump(self.als_cpu_, self.run_dir / "als_model_cpu.pkl")
                np.save(self.run_dir / "user_factors.npy", self.als_cpu_.user_factors)
                np.save(self.run_dir / "item_factors.npy", self.als_cpu_.item_factors)
            else:
                joblib.dump(self.als_gpu_, self.run_dir / "als_model_gpu.pkl")

            return

        except Exception as e:
            backend["backend"] = "cpu"
            backend["reason"] = repr(e)
            save_json(backend, self.run_dir / "als_backend.json")

        # 3) CPU ALS
        from implicit.als import AlternatingLeastSquares as CPUALS

        self.als_cpu_ = CPUALS(
            factors=self.factors,
            iterations=self.iterations,
            regularization=self.regularization,
            alpha=self.alpha,
            calculate_training_loss=False,
        )

        with threadpool_limits(limits=1, user_api="blas"):
            self.als_cpu_.fit(mat)

        joblib.dump(self.als_cpu_, self.run_dir / "als_model_cpu.pkl")
        np.save(self.run_dir / "user_factors.npy", self.als_cpu_.user_factors)
        np.save(self.run_dir / "item_factors.npy", self.als_cpu_.item_factors)



    # -------------------------
    # Recommend
    # -------------------------
    def recommend_for_user_id(self, user_id, k=None, filter_already_liked_items=True):
        if k is None:
            k = self.k_eval

        # cold-start
        if user_id not in self.user_id_to_index_:
            idxs = self.popular_item_indices_[:k].tolist()
            return [self.item_index_to_id_[i] for i in idxs]

        uidx = self.user_id_to_index_[user_id]

        # fallback (CPU model)
        if hasattr(self, "als_cpu_") and self.als_cpu_ is not None:
            user_items_row = self.user_item_matrix_[uidx]
            ids, scores = self.als_cpu_.recommend(
                userid=uidx,
                user_items=user_items_row,
                N=k,
                filter_already_liked_items=filter_already_liked_items
            )
            return [self.item_index_to_id_[i] for i in ids]

        # GPU recommend
        if hasattr(self, "als_gpu_"):
            user_items_row = self.user_item_matrix_[uidx]
            ids, scores = self.als_gpu_.recommend(
                userid=uidx,
                user_items=user_items_row,
                N=k,
                filter_already_liked_items=filter_already_liked_items
            )
            return [self.item_index_to_id_[i] for i in ids]
        
        # no model
        idxs = self.popular_item_indices_[:k].tolist()
        return [self.item_index_to_id_[i] for i in idxs]

    # -------------------------
    # Evaluate (dict) theo yêu cầu của bạn
    # -------------------------
    def evaluate(self, k=None, filter=True):
        if not hasattr(self, "user_history_"):
            raise RuntimeError("Model chưa fit. Hãy gọi fit() trước evaluate().")

        if self.df_valid.empty:
            return {"recall": 0.0, "hit": 0.0, "n_users_eval": 0}

        if k is None:
            k = self.k_eval

        df_val = self.df_valid[[self.user_col, self.item_col]].drop_duplicates().copy()

        item_cat_val = pd.Categorical(df_val[self.item_col], categories=self.item_index_to_id_)
        df_val["item_idx"] = item_cat_val.codes

        # bỏ item cold-start
        df_val = df_val[df_val["item_idx"] != -1]
        if df_val.empty:
            return {"recall": 0.0, "hit": 0.0, "n_users_eval": 0}

        user_to_gt = (
            df_val.groupby(self.user_col)["item_idx"]
                 .apply(lambda s: set(s.to_list()))
                 .to_dict()
        )

        recalls, hits = [], []

        iterator = user_to_gt.items()
        if self.use_tqdm:
            iterator = tqdm(iterator, total=len(user_to_gt), desc=f"Eval Stage1 @K={k} (filter={filter})", leave=False)

        for user_id, gt_items in iterator:
            # skip cold-start user
            if user_id not in self.user_id_to_index_:
                continue

            uidx = self.user_id_to_index_[user_id]
            hist = self.user_history_.get(uidx, set())

            relevant = (gt_items - hist) if filter else gt_items
            if len(relevant) == 0:
                continue

            # recommend (nếu filter=True thì filter_already_liked_items=True để nhất quán)
            rec_item_ids = self.recommend_for_user_id(user_id, k=k, filter_already_liked_items=filter)
            if not rec_item_ids:
                continue

            # map rec ids -> indices
            rec_idx = [self.item_id_to_index_.get(iid, -1) for iid in rec_item_ids]
            rec_idx = [x for x in rec_idx if x != -1]
            if not rec_idx:
                continue

            inter = set(rec_idx) & relevant
            recalls.append(len(inter) / len(relevant))
            hits.append(1.0 if len(inter) > 0 else 0.0)

        if len(recalls) == 0:
            return {"recall": 0.0, "hit": 0.0, "n_users_eval": 0}

        metrics = {
            "recall": float(np.mean(recalls)),
            "hit": float(np.mean(hits)),
            "n_users_eval": len(recalls),
        }
        save_json(metrics, self.run_dir / f"metrics_eval_k{k}_filter{int(filter)}.json")
        return metrics

    # -------------------------
    # sklearn score: return float
    # -------------------------
    def score(self, X=None, y=None):
        return self.evaluate(k=self.k_eval, filter=self.eval_filter)["recall"]

    # -------------------------
    # Fit end-to-end
    # -------------------------
    def fit(self, X=None, y=None):
        self._save_config()

        ui = self._build_user_item_matrix()
        # persist thêm thống kê cơ bản
        save_json(
            {"n_users": int(ui.shape[0]), "n_items": int(ui.shape[1]), "nnz": int(ui.nnz)},
            self.run_dir / "matrix_stats.json"
        )

        self._fit_als(ui)

        # lưu “full estimator” (ưu tiên CPU model đã convert nếu có)
        joblib.dump(self, self.run_dir / "stage1_estimator.pkl")
        return self

### Train + lưu ngay + evaluate theo 2 chế độ filter

In [17]:
df_item_pd = df_item.to_pandas()

In [18]:
import os

# Khóa BLAS về 1 thread để tránh xung đột/bug/perf issue
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

# Optional: giảm vấn đề threadpoolctl dò thư viện
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [19]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

from threadpoolctl import threadpool_limits

# Ép BLAS = 1 thread ngay trong runtime (phòng trường hợp env không ăn)
threadpool_limits(limits=1, user_api="blas")


In [20]:
# df_item_pd = df_item.to_pandas()  # bạn đã có sẵn từ trước
# df_train, df_valid: pandas

stage1 = ALSStage1CF(
    df_train=df_train,
    df_valid=df_valid,
    df_item=df_item_pd,
    user_col="customer_id",
    item_col="item_id",
    weight_type="rel_freq",
    alpha_cat=0.5,
    alpha_age=0.5,
    # ALS
    factors=128,
    iterations=30,
    regularization=0.05,
    alpha=2.0,
    # BM25
    use_bm25=True,
    bm25_k1=100,
    bm25_b=0.8,
    # Eval
    k_eval=200,
    eval_filter=True,
    use_tqdm=True,
    save_dir=str(ARTIFACT_DIR),
)

stage1.fit()

m1 = stage1.evaluate(k=200, filter=True)
m2 = stage1.evaluate(k=200, filter=False)

print("Eval filter=True :", m1)
print("Eval filter=False:", m2)

print("Artifacts saved to:", stage1.run_dir)


100%|██████████| 30/30 [15:51<00:00, 31.71s/it]


Eval filter=True : {'recall': 0.03059264691081953, 'hit': 0.0843215301866937, 'n_users_eval': 410512}
Eval filter=False: {'recall': 0.04075076429224421, 'hit': 0.12170532366707612, 'n_users_eval': 498228}
Artifacts saved to: artifacts_stage1_als_gpu/20251212_160644


### Recommend thử cho 1 user

In [21]:
test_user = df_train["customer_id"].iloc[0]
recs = stage1.recommend_for_user_id(test_user, k=10, filter_already_liked_items=True)
print("user:", test_user)
print("top-10 recs:", recs[:10], "...")

user: 1028293
top-10 recs: ['3988000000019', '3988000000016', '3987000000013', '3987000000018', '0953000000096', '4026000000027', '3987000000014', '4013000000034', '3988000000013', '3987000000017'] ...


### Load lại model (estimator) từ đĩa

In [ ]:
# load full estimator
loaded = joblib.load(stage1.run_dir / "stage1_estimator.pkl")

# dùng lại evaluate/recommend
print(loaded.evaluate(k=200, filter=True))

# Stage 2

In [ ]:
# import numpy as np

# def precision_at_k(pred, gt, hist, filter_bought_items=True, K=10): # prediction, ground-truth, history items, candidate items
#     precisions = []
#     ideal_precs = []
#     ncold_start = 0
#     cold_start_users = []
#     nusers = len(gt.keys())
#     for user in gt.keys():
#         if (user not in hist) or (user not in pred):
#             ncold_start += 1
#             cold_start_users.append(user) # THINKING: để giảm cold start có thể tăng khoảng HISTORY
#             continue
#         gt_items = gt[user]
#         relevant_items = set(gt_items)
#         if filter_bought_items:
#             relevant_items -= set(hist[user])
#         # Compute precision@k
#         hits = len(set(pred[user][:K]) & relevant_items)
#         precisions.append(hits / K)
#     return np.mean(precisions), cold_start_users


## Import, load ground truth, chuẩn bị lịch sử (hist) cho Stage 2

In [ ]:
# import pickle
# import json
# import numpy as np
# import pandas as pd
# import polars as pl
# import lightgbm as lgb
# from sklearn.model_selection import train_test_split
# from tqdm.auto import tqdm

# # 1) Load ground truth tháng 01/2025
# gt_raw = pd.read_pickle("groundtruth.pkl")

# # Giả định gt_raw là dict: {user_id: [item1, item2, ...]}
# if isinstance(gt_raw, dict):
#     gt = gt_raw
# elif isinstance(gt_raw, pd.Series):
#     gt = gt_raw.to_dict()
# else:
#     # Nếu format khác, bạn in ra để chỉnh tay:
#     print("groundtruth.pkl format:", type(gt_raw))
#     # TODO: chỉnh parse cho đúng cấu trúc của bạn
#     raise ValueError("Không biết format groundtruth.pkl, cần chỉnh lại parsing.")

# # 2) Chuẩn bị lịch sử mua trước 2025-01-01 (hist) từ df_transaction (Polars)
# #    đây là lịch sử để filter_bought_items trong precision_at_k

# cutoff_test = pd.Timestamp("2025-01-01")

# df_trx_hist = (
#     df_transaction
#     .select(["customer_id", "item_id", "created_date"])
#     .drop_nulls(["customer_id", "item_id", "created_date"])
#     .with_columns(
#         pl.col("created_date").cast(pl.Datetime).alias("created_datetime")
#     )
#     .filter(pl.col("created_datetime") < cutoff_test)
# )

# df_trx_hist_pd = df_trx_hist.to_pandas()

# hist = (
#     df_trx_hist_pd
#     .groupby("customer_id")["item_id"]
#     .apply(lambda s: list(set(s.tolist())))
#     .to_dict()
# )

# print("Số user trong ground truth:", len(gt))
# print("Số user có lịch sử trước 2025-01-01:", len(hist))

Số user trong ground truth: 391900
Số user có lịch sử trước 2025-01-01: 2442306


## Sinh candidate từ Stage 1 và lưu xuống file

In [ ]:
# best_model = joblib.load("stage1_item_item_cf.pkl")

# K_cand = 1000  # số candidate Stage 1 per user cho Stage 2 (bạn có thể đổi 100/300/... tùy ý)

# stage1_candidates = {}

# for user_id in tqdm(gt.keys(), desc="Generate Stage1 candidates"):
#     cand_items = best_model.recommend_for_user_id(user_id, top_k=K_cand)
#     stage1_candidates[user_id] = cand_items

# # Lưu lại để reuse
# with open("stage1_candidates.pkl", "wb") as f:
#     pickle.dump(stage1_candidates, f)

# print("Số user có candidate:", len(stage1_candidates))


Generate Stage1 candidates: 100%|██████████| 391900/391900 [07:08<00:00, 915.33it/s] 


Số user có candidate: 391900


## Xây tập dữ liệu ranking cho LightGBM (Stage 2)

Chuẩn bị user/item feature

In [ ]:
# # Chuyển df_user, df_item sang pandas
# df_user_pd = df_user.to_pandas()
# df_item_pd = df_item.to_pandas()

# # Rút gọn các cột cần dùng (có thể thêm/bớt tùy bạn)
# user_feat_cols = [
#     "customer_id",
#     "gender",
#     "province",
#     "membership",
#     "user_age_days",
#     "days_since_install",
#     "days_since_last_sync",
# ]

# df_user_feat = df_user_pd[user_feat_cols].drop_duplicates(subset=["customer_id"])

# item_feat_cols = [
#     "item_id",
#     "price",
#     "category_l1",
#     "category_l2",
#     "brand",
#     "age_group_final",
#     # có thể thêm: "item_type", "gender_target_final"
# ]

# df_item_feat = df_item_pd[item_feat_cols].drop_duplicates(subset=["item_id"])


Build DataFrame ranking

In [ ]:
# rows = []

# for user_id, cand_items in tqdm(stage1_candidates.items(), desc="Build ranking rows"):
#     gt_items = set(gt.get(user_id, []))

#     for rank_pos, item_id in enumerate(cand_items):
#         label = 1 if item_id in gt_items else 0
#         rows.append(
#             {
#                 "customer_id": user_id,
#                 "item_id": item_id,
#                 "label": label,
#                 "stage1_rank": rank_pos,  # vị trí trong output Stage1
#             }
#         )

# df_rank = pd.DataFrame(rows)
# print("Số dòng ranking:", len(df_rank))
# print(df_rank.head())


Build ranking rows: 100%|██████████| 391900/391900 [01:44<00:00, 3740.83it/s]


Số dòng ranking: 277884431
   customer_id        item_id  label  stage1_rank
0      6515994  6665000000002      0            0
1      6515994  6665000000004      0            1
2      6515994  2803000000010      0            2
3      6515994  2793000000004      0            3
4      6515994  3052000000001      0            4


Join user/item features vào df_rank

In [ ]:
# df_rank = df_rank.merge(df_user_feat, on="customer_id", how="left")
# df_rank = df_rank.merge(df_item_feat, on="item_id", how="left")

# # Một số fillna cơ bản
# df_rank["price"] = df_rank["price"].fillna(0.0)
# df_rank["stage1_rank"] = df_rank["stage1_rank"].fillna(K_cand).astype(int)

# # With categorical columns as category dtype
# cat_cols = [
#     "gender",
#     "province",
#     "membership",
#     "category_l1",
#     "category_l2",
#     "brand",
#     "age_group_final",
# ]

# for c in cat_cols:
#     if c in df_rank.columns:
#         df_rank[c] = df_rank[c].astype("category")


## Tách train/valid cho LightGBM, train bằng GPU và lưu model

In [ ]:
# # Lấy danh sách user có label trong ranking
# users_all = df_rank["customer_id"].unique()

# train_users, valid_users = train_test_split(
#     users_all, test_size=0.2, random_state=42
# )

# # df_train_rank = df_rank[df_rank["customer_id"].isin(train_users)].reset_index(drop=True)
# # df_valid_rank = df_rank[df_rank["customer_id"].isin(valid_users)].reset_index(drop=True)

# # print("Train users:", len(train_users), "Valid users:", len(valid_users))
# # print("Train rows:", len(df_train_rank), "Valid rows:", len(df_valid_rank))


In [ ]:
# # 1) Xác định lại list feature
# feature_cols = [
#     c for c in df_rank.columns
#     if c not in ["label", "customer_id", "item_id"]
# ]

# # 2) Khai báo các cột categorical theo tên (nếu tồn tại trong df_rank)
# cat_cols = [
#     "gender",
#     "province",
#     "membership",
#     "category_l1",
#     "category_l2",
#     "brand",
#     "age_group_final",
# ]

# cat_feature_names = [c for c in cat_cols if c in feature_cols]

# # 3) Xử lý cột categorical: fill NA bằng '__MISSING__' rồi cast sang category
# for c in cat_feature_names:
#     # chuyển sang string, fill missing, sau đó cast về category
#     df_rank[c] = df_rank[c].astype("string").fillna("__MISSING__").astype("category")

# # 4) Các cột numeric: ép về số, fillna(0.0)
# numeric_cols = [col for col in feature_cols if col not in cat_feature_names]

# for col in numeric_cols:
#     df_rank[col] = pd.to_numeric(df_rank[col], errors="coerce")
#     df_rank[col] = df_rank[col].fillna(0.0)

# # Không dùng df_rank.fillna(0.0) toàn bảng nữa!
# print(df_rank[feature_cols].dtypes)

# # 6) Tách lại train/valid như trước (nếu bạn đã tách rồi, chỉ cần update df_train_rank, df_valid_rank)
# df_train_rank = df_rank[df_rank["customer_id"].isin(train_users)].reset_index(drop=True)
# df_valid_rank = df_rank[df_rank["customer_id"].isin(valid_users)].reset_index(drop=True)

# X_train = df_train_rank[feature_cols]
# y_train = df_train_rank["label"]

# X_valid = df_valid_rank[feature_cols]
# y_valid = df_valid_rank["label"]

# # 7) Tạo danh sách index cho categorical_feature
# cat_feature_indices = [feature_cols.index(c) for c in cat_feature_names]

# train_data = lgb.Dataset(
#     X_train,
#     label=y_train,
#     categorical_feature=cat_feature_indices,
#     free_raw_data=False,
# )

# valid_data = lgb.Dataset(
#     X_valid,
#     label=y_valid,
#     categorical_feature=cat_feature_indices,
#     free_raw_data=False,
# )

# print("Train users:", len(train_users), "Valid users:", len(valid_users))
# print("Train rows:", len(df_train_rank), "Valid rows:", len(df_valid_rank))


stage1_rank                int64
gender                  category
province                category
membership              category
user_age_days            float64
days_since_install       float64
days_since_last_sync     float64
price                    float64
category_l1             category
category_l2             category
brand                   category
age_group_final         category
dtype: object
Train users: 313520 Valid users: 78380
Train rows: 222202093 Valid rows: 55682338


In [ ]:
# params = {
#     "objective": "binary",
#     "metric": ["auc", "binary_logloss"],
#     "boosting_type": "gbdt",
#     "learning_rate": 0.05,
#     "num_leaves": 64,
#     "max_depth": -1,
#     "feature_fraction": 0.8,
#     "bagging_fraction": 0.8,
#     "bagging_freq": 1,
#     "min_data_in_leaf": 50,
#     "verbosity": -1,
#     # GPU
#     "device": "gpu",      # bản mới dùng "device_type"
#     "gpu_device_id": 7,      # nếu cần chỉ định GPU ID
# }

# evals_result = {}

# callbacks = [
#     lgb.early_stopping(stopping_rounds=50, verbose=True),
#     lgb.record_evaluation(evals_result),
#     lgb.log_evaluation(period=50),
# ]

# bst = lgb.train(
#     params,
#     train_data,
#     num_boost_round=200,
#     valid_sets=[train_data, valid_data],
#     valid_names=["train", "valid"],
#     callbacks=callbacks,
# )

# bst.save_model("lgb_stage2_ranking.txt", num_iteration=bst.best_iteration)

# print("Best iteration:", bst.best_iteration)

Training until validation scores don't improve for 50 rounds


In [ ]:
# K_eval_final = 10  # K cho precision@K

# pred = {}

# for user_id, cand_items in tqdm(stage1_candidates.items(), desc="Predict Stage2 scores"):
#     if len(cand_items) == 0:
#         continue

#     # Subset candidate rows cho user này từ df_rank
#     # (để đảm bảo feature engineering giống lúc train)
#     mask = (df_rank["customer_id"] == user_id) & (df_rank["item_id"].isin(cand_items))
#     df_user_cand = df_rank.loc[mask, ["customer_id", "item_id"] + feature_cols].copy()

#     if df_user_cand.empty:
#         continue

#     X_user = df_user_cand[feature_cols]
#     scores = bst.predict(X_user, num_iteration=bst.best_iteration)

#     df_user_cand["score"] = scores

#     # Sort theo score giảm dần, lấy top K_eval_final
#     df_user_cand_sorted = df_user_cand.sort_values("score", ascending=False)
#     top_items = df_user_cand_sorted["item_id"].tolist()[:K_eval_final]

#     pred[user_id] = top_items

# # Tính precision@K theo hàm bạn cung cấp
# prec, cold_users = precision_at_k(
#     pred=pred,
#     gt=gt,
#     hist=hist,
#     filter_bought_items=True,
#     K=K_eval_final,
# )

# print(f"Precision@{K_eval_final}: {prec:.4f}")
# print("Số user bị xem là cold-start theo hàm precision:", len(cold_users))
